![Week 5 Day 1](../../assets/banners/w5d1.png)

# Week 5 Day 1 - Solution: Three Ways to Answer Q3

**The exercise:** write three new versions of Sam's Q3 answer (misspelt, a bare list, and a confident answer about the wrong concept), predict each mark with the handbook principles, then grade them with the stand-in and with your real model.

Try the exercise yourself first: struggling a little is where the learning happens! Then compare with this one. Everything runs offline with stand-ins (🎭); the 🔑 cells use your real models.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("❌ OpenAI API Key not set. Most of today's cells still run offline; the few that call a model will")
    print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

In [ ]:
import json

from langchain_core.embeddings import DeterministicFakeEmbedding

from gradewise.config import DATA_DIR, OUTPUT_DIR, settings
from gradewise.console import success
from gradewise.evals.compat import import_pandas
from gradewise.ingestion import ingest
from gradewise.rag import get_retriever
from gradewise.schemas import ExamPaper, TranscribedAnswer

pd = import_pandas()  # 💡 = `import pandas as pd`, with a fix for locked-down Windows machines

EXAM_FILE = DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json"
exam = ExamPaper.model_validate_json(EXAM_FILE.read_text(encoding="utf-8"))
QUESTIONS = {q.id: q for q in exam.questions}
TRUTH = json.loads((DATA_DIR / "answer-sheets" / "synthetic" / "transcripts.json").read_text(encoding="utf-8"))

# 🆓 A keyword-only (BM25) copy of the CS201 notes: fake embeddings, real keyword search, zero cost
OFFLINE_DB = OUTPUT_DIR / "week5_offline_db"
fake_embeddings = DeterministicFakeEmbedding(size=64)
ingest(collection="cs201_offline", embeddings=fake_embeddings, persist_dir=OFFLINE_DB, quiet=True)
bm25 = get_retriever("cs201_offline", k=4, mode="bm25", embeddings=fake_embeddings, persist_dir=OFFLINE_DB)

success(
    "Ready",
    {
        "Exam": f"{exam.title} ({len(exam.questions)} questions, {exam.total_marks:g} marks)",
        "Students": f"{len(TRUTH)} synthetic scripts",
        "Offline KB": "cs201_offline (BM25)",
        "Grading model": settings.grading_model,
    },
)

In [ ]:
from gradewise.errors import GradeWiseError
from gradewise.grading import grade_answer
from gradewise.grading.fakes import KeywordGraderModel

q3 = QUESTIONS["Q3"]
VARIANTS = {
    "a) misspelt": "A dedlock is when two procceses are stuck because each one is waiting for a resorce the other one "
    "holds. Conditions: mutul exclution and circuler wait.",
    "b) bare list": "mutual exclusion, hold and wait, no preemption, circular wait",
    "c) wrong concept": "A deadlock is when a low-priority process waits for a very long time because the "
    "scheduler keeps choosing higher-priority processes, so it never gets the CPU. This is solved with ageing, "
    "which slowly raises its priority. Deadlock is very common in round robin scheduling.",
}

### Predictions, using the handbook

| Variant | c1 definition (2) | c2 + c3 conditions (1 + 1) | Predicted | Principle |
|---|---|---|---|---|
| a) misspelt | 2: the meaning is clear and complete (*waiting for a resource the other holds*) | 2: "mutul exclution", "circuler wait" are clearly the right terms | **4/4** | 1: mark the content, not the spelling |
| b) bare list | 0: no definition at all | 2: two (in fact four) correct conditions | **2/4** | 4 applies to *explain* questions; Q3 only asks to *name* the conditions, so the names earn their marks |
| c) wrong concept | 0: this is **starvation**, not deadlock | 0: no conditions named | **0/4** | 2: credit correct points only; length and confidence earn nothing |

In [ ]:
rows = []
for label, text in VARIANTS.items():
    answer = TranscribedAnswer(question_id="Q3", text=text, confidence=1.0)
    graded = grade_answer(q3, answer, llm=KeywordGraderModel(), retriever=bm25, exam=exam)  # 🎭 stand-in
    rows.append(
        {
            "variant": label,
            "predicted": {"a": 4, "b": 2, "c": 0}[label[0]],
            "🎭 stand-in": graded.awarded_marks,
            "per criterion": " · ".join(f"{c.criterion_id}={c.awarded:g}" for c in graded.criteria),
        }
    )
pd.DataFrame(rows)

### What the stand-in gets wrong, and why
- **a) misspelt:** the keyword matcher can't recognise "mutul exclution" or "dedlock", so the student loses marks for spelling: a direct breach of **principle 1**. (This is also why week 4 insists the transcription stays verbatim: the *grader* must be the one that forgives spelling, knowingly.)
- **b) bare list:** it happens to agree with the prediction, but for the wrong reason: it would give a list the same credit on a question that says *explain*.
- **c) wrong concept:** 0, as predicted, but again by luck: the starvation answer simply doesn't use the rubric's keywords. Add one sentence with "blocked", "resource" and "held by another" to it and the stand-in starts handing out definition marks for an answer that is still about the wrong concept. Keyword overlap is not understanding.

A real LLM examiner should land on (or very near) the predictions. Check with your model below, and note how often it agrees with itself across the three runs.

In [ ]:
try:
    for label, text in VARIANTS.items():
        answer = TranscribedAnswer(question_id="Q3", text=text, confidence=1.0)
        graded = grade_answer(q3, answer, retriever=bm25, exam=exam, consistency_runs=3)  # 🔑 3 runs, median
        flag = f"  ⚑ {graded.review_reason}" if graded.needs_review else ""
        print(f"{label:<17} {graded.awarded_marks:g}/4  confidence {graded.confidence:.2f}{flag}")
        for c in graded.criteria:
            print(f"     {c.criterion_id}: {c.awarded:g}  {c.justification}")
except GradeWiseError as err:
    err.show()

**Takeaway:** the principles are written for humans, and a good LLM examiner follows them because it understands *meaning*. The stand-in can't, and that gap is exactly what Day 5 measures.